<a href="https://colab.research.google.com/github/Reeva-17/Customer-Churn-Prediction/blob/main/notebooks/08_XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [96]:
# Install Library

!pip install -q xgboost

In [97]:
# Import Libraries

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import xgboost as xgb

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.model_selection import GridSearchCV

In [98]:
# Load Processed Data

github_path = "https://raw.githubusercontent.com/Reeva-17/Customer-Churn-Prediction/main/data/processed/"

X_train = pd.read_csv(github_path + "X_train.csv")
X_test = pd.read_csv(github_path + "X_test.csv")
y_train = pd.read_csv(github_path + "y_train.csv").squeeze()
y_test = pd.read_csv(github_path + "y_test.csv").squeeze()

In [99]:
# Verify Dataset

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (5625, 30)
X_test  : (1407, 30)
y_train : (5625,)
y_test  : (1407,)


In [100]:
xgb.XGBClassifier(

    objective="binary:logistic",

    random_state=42,

    eval_metric="logloss",

    use_label_encoder=False

)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [101]:
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

print(scale_pos_weight)

2.762541806020067


In [102]:
# Initialize Model

from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    scale_pos_weight=scale_pos_weight
)

In [103]:
# Random Search

from sklearn.model_selection import RandomizedSearchCV

param_dist = {

    "n_estimators":[200,300,400],

    "max_depth":[3,4,5],

    "learning_rate":[0.03,0.05,0.07],

    "subsample":[0.8,0.9,1.0],

    "colsample_bytree":[0.8,0.9,1.0],

    "gamma":[0,0.1,0.2],

    "min_child_weight":[1,3,5],

    "reg_alpha":[0,0.1],

    "reg_lambda":[1,2,3]

}

random_search = RandomizedSearchCV(

    estimator=xgb,

    param_distributions=param_dist,

    n_iter=50,

    scoring="f1",

    cv=5,

    random_state=42,

    verbose=2,

    n_jobs=-1

)

random_search.fit(X_train,y_train)

best_model = random_search.best_estimator_

print("Best Parameters:")
print(random_search.best_params_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters:
{'subsample': 0.8, 'reg_lambda': 3, 'reg_alpha': 0, 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 3, 'learning_rate': 0.07, 'gamma': 0.2, 'colsample_bytree': 0.9}


In [85]:
# Grid Search

grid = GridSearchCV(

    estimator=xgb_model,

    param_grid=param_grid,

    scoring="f1",

    cv=5,

    n_jobs=-1,

    verbose=2

)

grid.fit(X_train,y_train)

Fitting 5 folds for each of 972 candidates, totalling 4860 fits


GridSearchCV(cv=5,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=True,
                                     eval_metric='logloss', feature_types=None,
                                     feature_weights=None, gamma=None,
                                     grow_policy=None, importance_type=None,
                                     interaction_constraints...
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.8, 1.0], 'gamma': [0, 0.1, 0.3],
                         'learning_rate': [0.01, 0.05, 0.1],
                         'max_depth': [3, 4, 5], 'min_child_weight': [1, 3, 5],
                         'n_estimators': [100, 200, 300],
                         'subsample': [0.8, 1.0]},
             scoring='f1', verbose=2)

In [86]:
# Best Parameters

print("Best Parameters :")

print(grid.best_params_)

print()

print("Best F1 Score :",grid.best_score_)

Best Parameters :
{'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 5, 'n_estimators': 200, 'subsample': 0.8}

Best F1 Score : 0.6366086973673097


In [87]:
# Best Model

best_model = grid.best_estimator_

In [88]:
# Optimize Threshold

import numpy as np
from sklearn.metrics import f1_score

y_prob = best_model.predict_proba(X_test)[:,1]

thresholds = np.arange(0.30,0.71,0.01)

best_threshold = 0.5
best_f1 = 0

for t in thresholds:

    pred = (y_prob >= t).astype(int)

    score = f1_score(y_test,pred)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("Best Threshold :",round(best_threshold,2))
print("Best F1 :",round(best_f1,4))

y_pred = (y_prob >= best_threshold).astype(int)

Best Threshold : 0.6
Best F1 : 0.6419


In [89]:
# Evaluate Model

accuracy = accuracy_score(y_test,y_pred)

precision = precision_score(y_test,y_pred)

recall = recall_score(y_test,y_pred)

f1 = f1_score(y_test,y_pred)

roc_auc = roc_auc_score(y_test,y_prob)

print("Accuracy :",round(accuracy,4))

print("Precision:",round(precision,4))

print("Recall   :",round(recall,4))

print("F1 Score :",round(f1,4))

print("ROC AUC  :",round(roc_auc,4))

Accuracy : 0.7811
Precision: 0.5679
Recall   : 0.738
F1 Score : 0.6419
ROC AUC  : 0.8373


In [90]:
# Classification Report

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.89      0.80      0.84      1033
           1       0.57      0.74      0.64       374

    accuracy                           0.78      1407
   macro avg       0.73      0.77      0.74      1407
weighted avg       0.81      0.78      0.79      1407



In [91]:
# Confusion Matrix

cm = confusion_matrix(y_test,y_pred)

print(cm)

[[823 210]
 [ 98 276]]


In [92]:
# Check Overfitting

train_pred = best_model.predict(X_train)

test_pred = best_model.predict(X_test)

train_f1 = f1_score(y_train,train_pred)

test_f1 = f1_score(y_test,test_pred)

print("Training F1 :",round(train_f1,4))

print("Testing F1  :",round(test_f1,4))

Training F1 : 0.6868
Testing F1  : 0.6146


In [93]:
# Model Summary

print(best_model)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=0,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=5, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)


In [94]:
# Save Model

import joblib

joblib.dump(best_model,"xgboost_model.pkl")

print("XGBoost model saved successfully!")

XGBoost model saved successfully!


In [95]:
# Download Model

from google.colab import files

files.download("xgboost_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>